In [ ]:
import pandas as pd, numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.0f}".format)

df = pd.read_csv("../data/raw/transactions-2026-11-04.csv", low_memory=False)
df["INSTANCE_DATE"] = pd.to_datetime(df["INSTANCE_DATE"], errors="coerce")
print(df.shape)

In [ ]:
for col in ["PROCEDURE_EN","IS_OFFPLAN_EN","IS_FREE_HOLD_EN","USAGE_EN",
            "PROP_TYPE_EN","PROP_SB_TYPE_EN","ROOMS_EN","PARKING"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))

print("\nProcedures inside each group:")
print(df.groupby("GROUP_EN")["PROCEDURE_EN"].value_counts())

In [ ]:
rep = df[df["TRANSACTION_NUMBER"].duplicated(keep=False)]
print("Rows sharing a transaction number:", len(rep),
      "| distinct numbers:", rep["TRANSACTION_NUMBER"].nunique())
print("Fully identical rows (all 22 columns):", df.duplicated().sum())
print("\nRows per repeated number:")
print(rep.groupby("TRANSACTION_NUMBER").size().value_counts().head())
print("\nGroups involved:")
print(rep["GROUP_EN"].value_counts())

sample_id = rep["TRANSACTION_NUMBER"].iloc[0]
print("\nOne repeated transaction, side by side:")
print(df[df["TRANSACTION_NUMBER"] == sample_id].T)

In [ ]:
sales = df[df["GROUP_EN"] == "Sales"]

print(sales["TRANS_VALUE"].describe())
print("\nSales with value <= 0:", (sales["TRANS_VALUE"] <= 0).sum())
print("Sales with ACTUAL_AREA <= 0:", (sales["ACTUAL_AREA"] <= 0).sum())

print("\n% of Sales with area <= 0, by property type:")
print(sales.groupby("PROP_TYPE_EN")["ACTUAL_AREA"].apply(lambda s: (s <= 0).mean() * 100))

print("\nTop 10 Sales by value:")
print(sales.nlargest(10, "TRANS_VALUE")[["TRANSACTION_NUMBER","INSTANCE_DATE",
      "AREA_EN","PROP_TYPE_EN","TRANS_VALUE","ACTUAL_AREA"]])

In [ ]:
areas = df["AREA_EN"]
print("Unique areas:", areas.nunique(),
      "| after trimming spaces and lowercasing:", areas.str.strip().str.lower().nunique())
print(sorted(areas.unique())[:40])